# Runbook 1.5.d: Faster, more accurate threat detection and response

**Skill:** Detect Attacks, Collaborate · **Maps to:** 1.5.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A SOC dashboard needs two key metrics to measure how well AI-assisted response is performing:

1. **MTTR** (Mean Time to Respond): the average seconds from detection to resolution,
   computed across all incidents in the dataset.
2. **auto_resolved**: the count of incidents that were closed automatically by an AI playbook
   (no human analyst required).

Implement `solve(incidents)` that reads a list of incident records and returns both metrics.

**Lesson objective.** Explain how AI reduces Mean Time to Respond (MTTR) in a security operations center, implement an incident-triage algorithm that prioritizes incidents by severity and computes response-time metrics, and articulate the difference between automated resolution and human-escalated response.

### What comes in
`incidents` -- a list of dicts, each with:
- `id` (str): unique incident identifier
- `detected_at` (int): Unix timestamp (seconds) when detected
- `resolved_at` (int): Unix timestamp (seconds) when resolved; always > detected_at
- `severity` (str): one of `"critical"`, `"high"`, `"medium"`, `"low"`
- `auto` (bool): True if auto-resolved, False if human-resolved

### What must come out
A dict with exactly two keys:
- `"mttr"` (float): mean response time in seconds across all incidents
- `"auto_resolved"` (int): count of incidents where `auto == True`

Formula:
  `mttr = sum(resolved_at - detected_at for each incident) / len(incidents)`

Return `{"mttr": 0.0, "auto_resolved": 0}` for empty input.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
incidents = [
    {"id": "INC-001", "detected_at": 1000, "resolved_at": 1120, "severity": "critical", "auto": False},
    {"id": "INC-002", "detected_at": 2000, "resolved_at": 2080, "severity": "high",     "auto": True},
]
# response times: 120s, 80s -> mean = 100.0; auto_resolved = 1
solve(incidents) -> {"mttr": 100.0, "auto_resolved": 1}

solve([]) -> {"mttr": 0.0, "auto_resolved": 0}

incidents = [
    {"id": "INC-003", "detected_at": 0, "resolved_at": 300, "severity": "low", "auto": True},
]
solve(incidents) -> {"mttr": 300.0, "auto_resolved": 1}
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(incidents: list[dict]) -> dict
```

#### Input (`incidents`)
A list of incident dicts. Each dict has exactly these keys:

| Key           | Type  | Description                                                          |
|---------------|-------|----------------------------------------------------------------------|
| `id`          | str   | Unique incident identifier string (e.g. `"INC-001"`)                |
| `detected_at` | int   | Unix timestamp (seconds) when the incident was detected              |
| `resolved_at` | int   | Unix timestamp (seconds) when the incident was resolved; always > detected_at |
| `severity`    | str   | One of `"critical"`, `"high"`, `"medium"`, `"low"`                  |
| `auto`        | bool  | `True` if the incident was resolved automatically; `False` if human-resolved |

Example single entry:
```python
{"id": "INC-001", "detected_at": 1000, "resolved_at": 1120, "severity": "critical", "auto": False}
```

#### Output (`result`)
A dict with exactly two keys:

| Key            | Type  | Description                                                           |
|----------------|-------|-----------------------------------------------------------------------|
| `"mttr"`       | float | Mean time to respond in seconds, averaged across ALL incidents.       |
| `"auto_resolved"` | int | Count of incidents where `auto == True`.                           |

`mttr` is computed as:
  `sum(resolved_at - detected_at for each incident) / len(incidents)`

If `incidents` is empty, return `{"mttr": 0.0, "auto_resolved": 0}`.

The `mttr` value is a Python `float` (may be a whole number like `120.0`).
The `auto_resolved` value is a Python `int`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- `incidents` is a list (may be empty).
- `resolved_at > detected_at` always; all timestamps are non-negative integers.
- `severity` is always one of the four valid strings; implementations need not validate it.
- No incident has a negative response time.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 1.5.d. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List, Dict, Any


def solve(incidents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Compute MTTR and auto-resolved count for a list of SOC incidents.

    Args:
        incidents: list of dicts with keys:
            id (str), detected_at (int), resolved_at (int),
            severity (str), auto (bool).
    Returns:
        {"mttr": float, "auto_resolved": int}
        mttr = mean(resolved_at - detected_at) across all incidents.
        auto_resolved = count of incidents where auto is True.
        Return {"mttr": 0.0, "auto_resolved": 0} for empty input.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() -- see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def _inc(id_, detected, resolved, severity="high", auto=False):
    return {"id": id_, "detected_at": detected, "resolved_at": resolved,
            "severity": severity, "auto": auto}


def test_empty_input():
    result = impl.solve([])
    assert result == {"mttr": 0.0, "auto_resolved": 0}


def test_single_incident_manual():
    result = impl.solve([_inc("INC-001", 1000, 1120, "critical", False)])
    assert result["mttr"] == 120.0
    assert result["auto_resolved"] == 0


def test_single_incident_auto():
    result = impl.solve([_inc("INC-002", 2000, 2060, "low", True)])
    assert result["mttr"] == 60.0
    assert result["auto_resolved"] == 1


def test_two_incidents_mean():
    incidents = [
        _inc("INC-003", 1000, 1120, "critical", False),
        _inc("INC-004", 2000, 2080, "high", True),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 100.0
    assert result["auto_resolved"] == 1


def test_all_auto_resolved():
    incidents = [
        _inc("INC-005", 0, 30, "low", True),
        _inc("INC-006", 0, 60, "medium", True),
        _inc("INC-007", 0, 90, "high", True),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 60.0
    assert result["auto_resolved"] == 3


def test_no_auto_resolved():
    incidents = [
        _inc("INC-010", 100, 700, "critical", False),
        _inc("INC-011", 200, 500, "high", False),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 450.0
    assert result["auto_resolved"] == 0


def test_mttr_is_float():
    result = impl.solve([_inc("INC-020", 0, 100, "medium", False)])
    assert isinstance(result["mttr"], float)


def test_auto_resolved_is_int():
    result = impl.solve([_inc("INC-021", 0, 50, "low", True)])
    assert isinstance(result["auto_resolved"], int)


def test_result_has_exactly_two_keys():
    result = impl.solve([_inc("INC-030", 0, 200)])
    assert set(result.keys()) == {"mttr", "auto_resolved"}


def test_non_integer_mean():
    incidents = [
        _inc("A", 0, 10),
        _inc("B", 0, 20),
        _inc("C", 0, 30),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 20.0


def test_fractional_mean():
    incidents = [
        _inc("X", 1000, 1010),
        _inc("Y", 2000, 2011),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 10.5


def test_large_dataset():
    incidents = [_inc(f"INC-{i}", i * 1000, i * 1000 + 300, "high", i % 2 == 0)
                 for i in range(100)]
    result = impl.solve(incidents)
    assert result["mttr"] == 300.0
    assert result["auto_resolved"] == 50


def test_mixed_severities_all_auto():
    incidents = [
        _inc("M1", 0, 45, "low", True),
        _inc("M2", 0, 45, "medium", True),
        _inc("M3", 0, 45, "high", True),
        _inc("M4", 0, 45, "critical", False),
    ]
    result = impl.solve(incidents)
    assert result["mttr"] == 45.0
    assert result["auto_resolved"] == 3


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     {"mttr": 0.0, "auto_resolved": 0}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it